# 01. 데이터 탐색 (Phase 1-1)

`data/raw/` 의 RFP 파일과 메타데이터 CSV 를 한 번 훑어보는 일회성 노트북이다. 결과 해석은 `docs/research/2026-11-11-데이터-탐색.md` 에 정리했다.

- 확인 항목: 확장자별 개수·용량, PDF 쪽수/스캔 의심, HWP 파싱 성공 여부(글자 수·표 개수·에러), 메타데이터 컬럼 요약, 파일 ↔ 메타데이터 매칭
- 문서 본문은 출력하지 않는다 (숫자와 짧은 발췌만).
- HWP 는 파서가 일부 파일에서 끝나지 않으므로 파일마다 **별도 프로세스 + 60초 제한**으로 파싱한다 (`docs/trials/2026-11-11-hwp-파서-무한루프-원인.md`).

In [1]:
import json
import subprocess
import sys
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import pymupdf

from rfp_agent.config import load_config

warnings.filterwarnings("ignore")
pymupdf.TOOLS.mupdf_display_errors(False)  # MuPDF 경고는 아래 PDF 절에서 따로 모은다
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 50)

cfg = load_config("configs/default.yaml")
RAW = Path(cfg["paths"]["raw_dir"])
META = Path(cfg["paths"]["metadata_file"])
HWP_TIMEOUT_SEC = 60
# 스캔 PDF 의심 기준: 쪽당 평균 글자 수 < 50, 또는 글자 20자 미만인 쪽이 절반 이상
SCAN_AVG, SCAN_EMPTY_CHARS, SCAN_EMPTY_RATIO = 50, 20, 0.5


def nfc(s: str) -> str:
    """macOS 파일명(NFD)과 CSV(NFC) 비교를 위해 정규화한다."""
    return unicodedata.normalize("NFC", s)


def kind_of(name: str) -> str:
    return Path(nfc(name).lower()).suffix.lstrip(".") or "(none)"


print("raw_dir:", RAW, "| metadata:", META.name)

raw_dir: /Users/codeit/dev/rfp-agent-yule/data/raw | metadata: data_list.csv


## 1. 파일 목록

In [2]:
files = sorted(p for p in RAW.iterdir() if p.is_file() and p != META)
inv = pd.DataFrame(
    {
        "name": [nfc(p.name) for p in files],
        "kind": [kind_of(p.name) for p in files],
        "size_mb": [round(p.stat().st_size / 1e6, 3) for p in files],
    }
)
print("파일 수:", len(inv), dict(Counter(inv.kind)))
print("용량 합계 MB:", round(inv.size_mb.sum(), 1))
inv.size_mb.describe().round(2).to_frame("size_mb")

파일 수: 100 {'hwp': 96, 'pdf': 4}
용량 합계 MB: 170.8


       size_mb
count   100.00
mean      1.71
std       2.74
min       0.15
25%       0.56
50%       0.97
75%       1.62
max      24.11

## 2. PDF: 쪽수 / 글자 수 / 스캔 의심 / 표

In [3]:
def inspect_pdf(path: Path) -> dict:
    pymupdf.TOOLS.reset_mupdf_warnings()
    with pymupdf.open(path) as doc:
        chars = [len(pg.get_text("text").strip()) for pg in doc]
        n_images = sum(1 for pg in doc if pg.get_images())
        tables, fail = 0, 0
        for pg in doc:
            try:
                tables += len(pg.find_tables().tables)
            except Exception:  # noqa: BLE001 - 표 탐지 실패는 개수만 센다
                fail += 1
        repaired = doc.is_repaired
    n = len(chars)
    avg = sum(chars) / n
    empty = sum(c < SCAN_EMPTY_CHARS for c in chars)
    warn = pymupdf.TOOLS.mupdf_warnings()
    return {
        "name": nfc(path.name)[:36],
        "pages": n,
        "chars_per_page": round(avg, 1),
        "empty_pages": empty,
        "pages_with_images": n_images,
        "tables": tables,
        "table_fail_pages": fail,
        "repaired": repaired,
        "mupdf_warn_lines": len([w for w in warn.splitlines() if w.strip()]),
        "scanned_suspect": avg < SCAN_AVG or empty / n >= SCAN_EMPTY_RATIO,
    }


pdf_df = pd.DataFrame([inspect_pdf(p) for p in files if kind_of(p.name) == "pdf"])
print(
    "PDF",
    len(pdf_df),
    "건, 총",
    pdf_df.pages.sum(),
    "쪽, 스캔 의심",
    int(pdf_df.scanned_suspect.sum()),
    "건",
)
pdf_df

PDF 4 건, 총 570 쪽, 스캔 의심 0 건


                                   name  pages  chars_per_page  empty_pages  pages_with_images  tables  table_fail_pages  repaired  mupdf_warn_lines  \
0        고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf    297           707.4            1                297     310                 0     False                 0   
1  기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.     49           928.8            1                 21      29                 0     False                 2   
2  서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템     149           778.7            0                  6     138                 0     False                 2   
3  서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고     75          1597.4            1                 11     127                 0     False                16   

   scanned_suspect  
0            False  
1            False  
2            False  
3            False  

## 3. HWP: 파싱 가능 여부 (파일마다 별도 프로세스, 60초 제한)

In [4]:
WORKER = r"""
import sys, json, time
from pathlib import Path
from hwp_hwpx_parser import Reader
t0 = time.perf_counter()
try:
    with Reader(Path(sys.argv[1])) as r:
        text = r.text
        tables = r.tables
        out = dict(opens=True, file_type=str(r.file_type), encrypted=bool(r.is_encrypted),
                   chars=len(text), chars_no_ws=len("".join(text.split())), tables=len(tables))
except Exception as e:
    out = dict(opens=False, error=f"{type(e).__name__}: {e}"[:300])
out["seconds"] = round(time.perf_counter() - t0, 2)
print(json.dumps(out))
"""


def inspect_hwp(path: Path) -> dict:
    t0 = time.perf_counter()
    try:
        res = subprocess.run(
            [sys.executable, "-W", "ignore", "-c", WORKER, str(path)],
            check=False,
            capture_output=True,
            text=True,
            timeout=HWP_TIMEOUT_SEC,
        )
        return json.loads(res.stdout.strip().splitlines()[-1])
    except subprocess.TimeoutExpired:
        return {
            "opens": False,
            "error": f"Timeout {HWP_TIMEOUT_SEC}s",
            "timeout": True,
            "seconds": round(time.perf_counter() - t0, 2),
        }
    except Exception as e:  # noqa: BLE001
        return {"opens": False, "error": f"{type(e).__name__}: {e}"[:300]}


rows = []
for p in files:
    if kind_of(p.name) in ("hwp", "hwpx"):
        rows.append({"name": nfc(p.name), **inspect_hwp(p)})
hwp_df = pd.DataFrame(rows)
ok = hwp_df[hwp_df.opens]
print("HWP", len(hwp_df), "건 중 파싱 성공", len(ok), "건, 실패", len(hwp_df) - len(ok), "건")
print("file_type:", dict(Counter(ok.file_type)), "| 암호화:", int(ok.encrypted.sum()))
print("표 합계:", int(ok.tables.sum()))
ok[["chars", "chars_no_ws", "tables", "seconds"]].describe().round(1)

HWP 96 건 중 파싱 성공 95 건, 실패 1 건
file_type: {'FileType.HWP5': 95} | 암호화: 0
표 합계: 11793


          chars  chars_no_ws  tables  seconds
count      95.0         95.0    95.0     95.0
mean    80573.9      57006.1   124.1      0.2
std     24397.5      17538.1    41.8      0.1
min     48980.0      35079.0    12.0      0.1
25%     64727.0      45584.5    94.0      0.2
50%     74845.0      53115.0   121.0      0.2
75%     88503.0      62525.0   144.5      0.2
max    167248.0     119831.0   263.0      0.5

In [5]:
# 실패한 파일 (본문은 출력하지 않음)
hwp_df[~hwp_df.opens][["name", "error", "seconds"]]

                                           name        error  seconds
60  케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp  Timeout 60s    60.01

## 4. 메타데이터 CSV

In [6]:
meta = pd.read_csv(META, encoding="utf-8-sig")
print("행:", len(meta), "열:", len(meta.columns))
summary = pd.DataFrame(
    {
        "dtype": meta.dtypes.astype(str),
        "null_ratio": meta.isna().mean().round(3),
        "n_unique": meta.nunique(),
    }
)
summary

행: 100 열: 12


             dtype  null_ratio  n_unique
공고 번호          str        0.18        82
공고 차수      float64        0.18         3
사업명            str        0.00        99
사업 금액      float64        0.01        82
발주 기관          str        0.00        87
공개 일자          str        0.00        99
입찰 참여 시작일      str        0.26        73
입찰 참여 마감일      str        0.08        87
사업 요약          str        0.00       100
파일형식           str        0.00         2
파일명            str        0.00       100
텍스트            str        0.00       100

In [7]:
# 샘플 행 (긴 문자열은 40자로 자르고, 본문성 컬럼인 '텍스트'는 제외)
sample = meta.drop(columns=["텍스트"]).head(3).copy()
for c in sample.select_dtypes(include=["object", "string"]).columns:
    sample[c] = sample[c].astype(str).str.replace("\n", " ").str.slice(0, 40)
sample.T

                                                  0                                         1                                         2
공고 번호                                   20241001798                               20241002912                               20240827859
공고 차수                                           0.0                                       0.0                                       0.0
사업명        한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화        2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선                EIP3.0 고압가스 안전관리 시스템 구축 용역
사업 금액                                   130000000.0                               129300000.0                                40000000.0
발주 기관                                          한영대학                                    한국연구재단                                 한국생산기술연구원
공개 일자                           2024-10-04 13:51:23                       2024-10-04 15:01:52                       2024-08-28 11:31:02
입찰 참여 시작일                                       

In [8]:
amount = meta["사업 금액"]
print(
    "사업 금액: 결측",
    int(amount.isna().sum()),
    "| 0원",
    int((amount == 0).sum()),
    "| 중앙값",
    f"{amount.median():,.0f}",
    "| 최대",
    f"{amount.max():,.0f}",
)
for c in ["공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]:
    d = pd.to_datetime(meta[c], errors="coerce")
    print(f"{c}: {d.min()} ~ {d.max()}")
print(
    "공개 일자가 2024년 이전:", int((pd.to_datetime(meta["공개 일자"]) < "2024-01-01").sum()), "건"
)
print(
    "공고 번호 결측:",
    int(meta["공고 번호"].isna().sum()),
    "| 공고 차수 분포:",
    meta["공고 차수"].value_counts(dropna=False).to_dict(),
)
print("발주 기관 고유 수:", meta["발주 기관"].nunique())

사업 금액: 결측 1 | 0원 6 | 중앙값 172,700,000 | 최대 14,107,009,000
공개 일자: 2021-10-08 00:00:00 ~ 2025-02-11 13:32:46
입찰 참여 시작일: 2024-02-21 10:00:00 ~ 2025-02-11 14:00:00
입찰 참여 마감일: 2024-02-26 10:00:00 ~ 2025-03-10 11:00:00
공개 일자가 2024년 이전: 2 건
공고 번호 결측: 18 | 공고 차수 분포: {0.0: 76, nan: 18, 1.0: 4, 2.0: 2}
발주 기관 고유 수: 87


## 5. 파일 ↔ 메타데이터 매칭 (조인 키 = 파일명, NFC 정규화)

In [9]:
file_names = set(inv.name)
csv_names = [nfc(str(n)) for n in meta["파일명"]]
csv_set = set(csv_names)
print("CSV 행", len(meta), "| 고유 파일명", len(csv_set), "| 실제 파일", len(file_names))
print("일치:", len(csv_set & file_names))
print("CSV 에만 있음:", sorted(csv_set - file_names))
print("파일에만 있음:", sorted(file_names - csv_set))
ext_mismatch = [n for n, k in zip(csv_names, meta["파일형식"]) if kind_of(n) != k]
print("CSV 파일형식 컬럼 ≠ 실제 확장자:", len(ext_mismatch), "건")

CSV 행 100 | 고유 파일명 100 | 실제 파일 100
일치: 100
CSV 에만 있음: []
파일에만 있음: []
CSV 파일형식 컬럼 ≠ 실제 확장자: 0 건


## 6. CSV `텍스트` 컬럼 vs 파서 본문 길이

ADR 0004 의 근거 데이터. 본문은 출력하지 않고 길이만 비교한다. (CSV 텍스트는 대부분 문서 목차라 사용하지 않는다.)

In [10]:
csv_len = dict(zip(csv_names, meta["텍스트"].fillna("").str.len()))
cmp = ok.assign(csv_text_chars=ok.name.map(csv_len))
cmp["parsed_over_csv"] = (cmp.chars / cmp.csv_text_chars).round(1)
print(
    "CSV 텍스트 글자 수 중앙값:",
    int(meta["텍스트"].fillna("").str.len().median()),
    "| 파서 본문 글자 수 중앙값:",
    int(ok.chars.median()),
    "| 비율 중앙값:",
    cmp.parsed_over_csv.median(),
)
cmp[["csv_text_chars", "chars", "parsed_over_csv"]].describe().round(1)

CSV 텍스트 글자 수 중앙값: 2583 | 파서 본문 글자 수 중앙값: 74845 | 비율 중앙값: 27.8


       csv_text_chars     chars  parsed_over_csv
count            95.0      95.0             95.0
mean           3890.5   80573.9             64.9
std            3721.5   24397.5            116.0
min              89.0   48980.0              3.2
25%            1226.5   64727.0             14.4
50%            2792.0   74845.0             27.8
75%            5844.0   88503.0             62.8
max           18335.0  167248.0            751.7

## 7. 요약 저장

In [11]:
report = {
    "files": len(inv),
    "by_kind": dict(Counter(inv.kind)),
    "size_mb_total": round(float(inv.size_mb.sum()), 1),
    "pdf": {
        "count": len(pdf_df),
        "pages": int(pdf_df.pages.sum()),
        "scanned_suspects": int(pdf_df.scanned_suspect.sum()),
        "tables": int(pdf_df.tables.sum()),
    },
    "hwp": {
        "count": len(hwp_df),
        "parsed": len(ok),
        "failed": hwp_df[~hwp_df.opens].name.tolist(),
        "chars_median": int(ok.chars.median()),
        "tables_total": int(ok.tables.sum()),
        "tables_median": int(ok.tables.median()),
    },
    "metadata": {
        "rows": len(meta),
        "columns": list(meta.columns),
        "null_ratio": meta.isna().mean().round(3).to_dict(),
    },
    "matching": {
        "matched": len(csv_set & file_names),
        "csv_only": len(csv_set - file_names),
        "file_only": len(file_names - csv_set),
        "ext_mismatch": len(ext_mismatch),
    },
}
out = Path(cfg["paths"]["processed_dir"]) / "inspect_report.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(
    json.dumps({k: report[k] for k in ["files", "by_kind", "pdf", "matching"]}, ensure_ascii=False)
)
print("저장:", out)

{"files": 100, "by_kind": {"hwp": 96, "pdf": 4}, "pdf": {"count": 4, "pages": 570, "scanned_suspects": 0, "tables": 604}, "matching": {"matched": 100, "csv_only": 0, "file_only": 0, "ext_mismatch": 0}}
저장: /Users/codeit/dev/rfp-agent-yule/data/processed/inspect_report.json
